# TF-IDF + LR + XGBOOST Blend

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
import xgboost as xgb
from tqdm import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 1. Data loading and EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option')
plt.ylabel('Count')
plt.show()

train_df['opt_len'] = train_df[LABELS].apply(lambda row: row.str.len().mean(), axis=1)
plt.figure(figsize=(6,4))
sns.histplot(train_df['opt_len'], bins=30, kde=True)
plt.title('Average Option Length Distribution')
plt.xlabel('Avg characters')
plt.show()

def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['option_set'] = train_df.apply(option_set_key, axis=1)
dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Number of duplicate option-sets in train: {dup_count}")

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)

## 2. Data preprocessing and splitting

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)}")

## 3. Feature Engineering (TF‑IDF based)

In [ ]:
fit_texts = []
for _, row in train_split.iterrows():
    fit_texts.append(clean_prompt(row['prompt']))
    for l in LABELS:
        fit_texts.append(clean_text(row[l]))

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)
tfidf.fit(fit_texts)

def extract_features(df):
    rows = []
    for _, row in df.iterrows():
        prompt = clean_prompt(row['prompt'])
        pv = tfidf.transform([prompt])
        cos, clen, wlen, ov, jac, neg, num = [], [], [], [], [], [], []
        for l in LABELS:
            opt = clean_text(row[l])
            ov_vec = tfidf.transform([opt])
            cos.append(float(cosine_similarity(pv, ov_vec)[0,0]))
            clen.append(len(row[l]))
            wlen.append(len(opt.split()))
            ov.append(len(set(prompt.split()) & set(opt.split())))
            jac.append(ov[-1] / (len(set(prompt.split()) | set(opt.split())) + 1))
            neg.append(1 if re.search(r'\bnot\b|\bno\b|\bnever\b|\bcannot\b|\bnone\b', opt) else 0)
            num.append(1 if re.search(r'\d', row[l]) else 0)
        rows.append((cos, clen, wlen, ov, jac, neg, num))
    
    base = []
    for r in rows:
        cos_rank = (-np.array(r[0])).argsort().argsort().astype(float)
        cos_z = (np.array(r[0]) - np.mean(r[0])) / (np.std(r[0]) + 1e-6)
        len_z = (np.array(r[1]) - np.mean(r[1])) / (np.std(r[1]) + 1e-6)
        ov_z  = (np.array(r[3]) - np.mean(r[3])) / (np.std(r[3]) + 1e-6)
        cos_std, cos_max, cos_min = np.std(r[0]), np.max(r[0]), np.min(r[0])
        feat = []
        for i in range(5):
            feat += [r[0][i], r[1][i], r[2][i], r[3][i], r[4][i], r[5][i], r[6][i], cos_rank[i]]
        feat += list(cos_z) + list(len_z) + list(ov_z) + [cos_std, cos_max, cos_min]
        base.append(feat)
    return np.array(base)

train_feats = extract_features(train_split)
val_feats   = extract_features(val_split)
test_feats  = extract_features(test_df)

## 4. Metric: MAP@3

In [ ]:
def map3_from_probs(probs, true_idx):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    true_lab = [LABELS[i] for i in true_idx]
    pred_lab = [" ".join(LABELS[j] for j in row) for row in top3]
    score = 0.0
    for t, p in zip(true_lab, pred_lab):
        for i, c in enumerate(p.split()[:3]):
            if c == t:
                score += 1.0/(i+1); break
    return score / len(true_idx)

## 5. Cross‑Validation: Logistic Regression & XGBoost

In [ ]:
GROUP_NAME = "xgb_lr_blend"   

gkf = GroupKFold(n_splits=5)
oof_lr = np.zeros((len(train_split), 5))
oof_xgb = np.zeros((len(train_split), 5))
val_lr_list, val_xgb_list = [], []
test_lr_list, test_xgb_list = [], []

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'max_depth': 5,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': SEED
}

for fold, (tr_i, va_i) in enumerate(gkf.split(train_feats, y_tr, groups)):
    wandb.init(
        project="smart-mcq-solver",
        entity="23f2004192-dl-genai-project",
        group=GROUP_NAME,
        job_type="cv_fold",
        tags=["tfidf", "ensemble"],
        name=f"fold_{fold}"
    )

    X_tr, y_tr_f = train_feats[tr_i], y_tr[tr_i]
    X_va, y_va_f = train_feats[va_i], y_tr[va_i]

    # Logistic Regression
    lr = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                            max_iter=1000, random_state=SEED, C=1.0)
    lr.fit(X_tr, y_tr_f)
    lr_tr_probs   = lr.predict_proba(X_tr)
    lr_va_probs   = lr.predict_proba(X_va)
    lr_val_probs  = lr.predict_proba(val_feats)
    lr_test_probs = lr.predict_proba(test_feats)
    oof_lr[va_i] = lr_va_probs
    val_lr_list.append(lr_val_probs)
    test_lr_list.append(lr_test_probs)

    lr_preds      = lr_va_probs.argmax(axis=1)
    lr_map3       = map3_from_probs(lr_va_probs, y_va_f)
    lr_acc        = accuracy_score(y_va_f, lr_preds)
    lr_f1         = f1_score(y_va_f, lr_preds, average='macro')
    lr_train_loss = log_loss(y_tr_f, lr_tr_probs, labels=list(range(5)))
    lr_val_loss   = log_loss(y_va_f, lr_va_probs, labels=list(range(5)))

    # XGBoost
    dtrain = xgb.DMatrix(X_tr, label=y_tr_f)
    dval   = xgb.DMatrix(X_va, label=y_va_f)
    evals_result = {}
    bst = xgb.train(xgb_params, dtrain, num_boost_round=300,
                    evals=[(dtrain, 'train'), (dval, 'eval')],
                    early_stopping_rounds=20, evals_result=evals_result,
                    verbose_eval=False)

    # epoch-by-epoch (boosting round) loss progression
    train_logloss = evals_result['train']['mlogloss']
    eval_logloss  = evals_result['eval']['mlogloss']
    for round_i, (tr_l, va_l) in enumerate(zip(train_logloss, eval_logloss)):
        wandb.log({
            "xgb/train_mlogloss": tr_l,
            "xgb/val_mlogloss": va_l,
            "epoch": round_i
        })

    xgb_va_probs   = bst.predict(dval)
    xgb_val_probs  = bst.predict(xgb.DMatrix(val_feats))
    xgb_test_probs = bst.predict(xgb.DMatrix(test_feats))
    oof_xgb[va_i] = xgb_va_probs
    val_xgb_list.append(xgb_val_probs)
    test_xgb_list.append(xgb_test_probs)

    xgb_preds = xgb_va_probs.argmax(axis=1)
    xgb_map3  = map3_from_probs(xgb_va_probs, y_va_f)
    xgb_acc   = accuracy_score(y_va_f, xgb_preds)
    xgb_f1    = f1_score(y_va_f, xgb_preds, average='macro')

    print(f"Fold {fold} | LR  -> MAP3: {lr_map3:.4f}  Acc: {lr_acc:.4f}  F1: {lr_f1:.4f}")
    print(f"Fold {fold} | XGB -> MAP3: {xgb_map3:.4f}  Acc: {xgb_acc:.4f}  F1: {xgb_f1:.4f}")

    # final per-fold summary metrics 
    wandb.log({
        "lr/val_map3": lr_map3,
        "lr/val_accuracy": lr_acc,
        "lr/val_f1": lr_f1,
        "lr/train_loss": lr_train_loss,
        "lr/val_loss": lr_val_loss,
        "xgb/val_map3": xgb_map3,
        "xgb/val_accuracy": xgb_acc,
        "xgb/val_f1": xgb_f1,
    })
    wandb.finish()

# Average fold predictions
val_lr  = np.mean(val_lr_list, axis=0)
val_xgb = np.mean(val_xgb_list, axis=0)
test_lr = np.mean(test_lr_list, axis=0)
test_xgb= np.mean(test_xgb_list, axis=0)

lr_val_map3  = map3_from_probs(val_lr, val_split['label'].values)
xgb_val_map3 = map3_from_probs(val_xgb, val_split['label'].values)
print(f"\nLR Val MAP3: {lr_val_map3:.4f}")
print(f"XGB Val MAP3: {xgb_val_map3:.4f}")

## 6. Blend (Weighted Average)

In [ ]:
val_true = val_split['label'].values
best_score = 0
best_w_lr, best_w_xgb = 0, 1

for w1 in np.arange(0, 1.05, 0.05):
    w2 = 1 - w1
    combined = w1 * val_lr + w2 * val_xgb
    score = map3_from_probs(combined, val_true)
    if score > best_score:
        best_score = score
        best_w_lr, best_w_xgb = w1, w2

print(f"Best weights: LR={best_w_lr:.2f}, XGB={best_w_xgb:.2f} -> Val MAP3: {best_score:.4f}")

blend_val_probs  = best_w_lr * val_lr + best_w_xgb * val_xgb
blend_test_probs = best_w_lr * test_lr + best_w_xgb * test_xgb

blend_preds  = np.argmax(blend_val_probs, axis=1)
ensemble_acc = accuracy_score(val_true, blend_preds)
ensemble_f1  = f1_score(val_true, blend_preds, average='macro')

# WandB Summary Run
wandb.init(
    project="smart-mcq-solver",
    entity="23f2004192-dl-genai-project",
    group=GROUP_NAME,
    job_type="ensemble",
    name="ensemble_summary",
    tags=["tfidf", "ensemble", "final"],
    config={
        "best_lr_weight": best_w_lr,
        "best_xgb_weight": best_w_xgb,
    }
)

wandb.log({
    "ensemble_val_map3": best_score,
    "ensemble_val_accuracy": ensemble_acc,
    "ensemble_val_f1": ensemble_f1,
    "best_lr_weight": best_w_lr,
    "best_xgb_weight": best_w_xgb,
})

wandb.finish()

## 7. Evaluation on Validation Set

In [ ]:
val_preds = np.argmax(blend_val_probs, axis=1)
acc = accuracy_score(val_true, val_preds)
f1 = f1_score(val_true, val_preds, average='macro')
print(f"Validation Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")

## 8. Final Submission

In [ ]:
top3 = np.argsort(-blend_test_probs, axis=1)[:, :3]
preds = [(test_df.iloc[i]['id'], " ".join(LABELS[j] for j in top3[i])) for i in range(len(test_df))]
sub = pd.DataFrame(preds, columns=['ID','Prediction']).sort_values('ID').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)
print("\nSubmission saved")
print(sub.head(10))